## **🚀PART-2: Node Onboarding(with policies) in a Cluster, Block Deployment (with policies)**

In this demo, we will cover how to onboard a node to a existing cluster, how to setup pre-check policy to Gateway and Cluster Controller, how to deploy a block with policies(health check and resource allocation)

### **Pre-requisites:**

- Watch Part-1:
    - understand how to create k8s cluster and attach the cluster to AIOS
    - Deploy a block and do inference on it
- Create a cluster with Master Node and if needed Worker nodes
- Install prerequisites on Worker Node:
    - Run the script for installing base packages
        - `sudo bash part-2_install_workernode.sh`([worker_install_script](cluster_setup/gpu_node/part-2_install_workernode.sh))
    - Install Nvidia Driver(if GPU available), Nvidia Container Runtime(If GPU available) etc.
        - Then install the Nvidia Container Toolkit by running the following this   `https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html`
    - **Important**: Once Node joined, you can replace toml file to /etc/containerd/config.toml


### **Policies of Control @Gateway and Cluster Controller**

- Let us write some policies to:
    - add_node policy to:
        - Cluster Controller Gateway
            - check in **[policies/GatewayPolicy](policies/GatewayPolicy)**
                - As Gateway is under controll of network owner:
                    - minimalistic policy to check minimum requirements of the network will be written by network administrator
        - Cluster Controller
            - check in **[policies/Cluster_Controller](policies/Cluster_Controller)**
                - Cluster admin will write policies to check minimum and maximum requirement of the cluster
                - like:
                    - minimum number of GPUs
                    - minimum memory
                    - minimum CPU cores
                    - maximum number of GPUs
                    - mamimum number of nodes
                    - etc.
    - more policies:
        - Cluster Controller Gateway:
            - `remove_node`, `add_cluster`, `remove_cluster`, `add_block`, `remove_block`, `update_cluster`, `scale_block`, `block_mgmt`, `cluster_mgmt`
        - Cluster Controller:
            - `remove_node`, `remove_vdag_controller`, `add_vdag_controller`, `add_block`, `remove_block`, `update_cluster`, `scale_block`, `block_mgmt`, `cluster_mgmt`


- **Process For Onboarding the Policy**:

    - Create a file `function.py` and place it in direcrectory `code`
    - You can place `requirements.txt` also in code directory.
    - The directory structure should look like this:
        ```
        code/
        ├── function.py
        └── requirements.txt
        ```

    - zip the code directory with `zip -r mypolicy.zip code`

    - Upload the zip file to AIOS Policy storage using the command: `bash upload.sh`
    
    - Registration of the needs to be done: `bash register.sh`

- **Onboard add_node for gateway**:

    - Upload the Zip file to AIOS Policy Storage:
    - Register the policy:

In [ ]:
%%bash
bash policies/GatewayPolicy/add_node/upload.sh

In [ ]:
%%bash 
bash policies/GatewayPolicy/add_node/register.bash

- **Onboard add_node for Cluster Controller**:

    - Upload the Zip file to AIOS Policy Storage:
    - Register the policy:

In [ ]:
%%bash
bash policies/Cluster_Controller/add_node/upload.sh

In [ ]:
%%bash 
bash policies/Cluster_Controller/add_node/register.bash

**To  GET PORT MAPPING wrt to Service**([Doc](https://docs.aigr.id/installation/installation/#deploying-registry-services))

In [ ]:
# 🔧 Configuration Setup - Run this cell first to set up shared variables
import os

# Set configuration variables that will be available across all cells
GATEWAY_URL = "MANAGEMENTMASTER:30600"
CLUSTER_ID = "gcp-demo-cluster"
CLUSTER_SYNC_URL = "DEMOCLUSTERMASTER:30501"
GLOBAL_CLUSTER_METRICS_DB = "MANAGEMENTMASTER:30202"
GLOBAL_BLOCK_METRICS_DB = "MANAGEMENTMASTER:30201"
PARSER_URL = "MANAGEMENTMASTER:30501"
GLOBAL_CLUSTER_DB = "MANAGEMENTMASTER:30101"
GLOBAL_TASK_DB_SERVICE = "MANAGEMENTMASTER:30108"
COMPONENT_REGISTRY_SERVICE = "MANAGEMENTMASTER:30112"
GLOBAL_BLOCKDB_SERVICE = "MANAGEMENTMASTER:30100"
#SERVER_URL = "10.10.10.10:5000"  # For other API calls

# Set environment variables for bash cells
os.environ['GATEWAY_URL'] = GATEWAY_URL
os.environ['CLUSTER_ID'] = CLUSTER_ID
os.environ['GLOBAL_CLUSTER_METRICS_DB'] = GLOBAL_CLUSTER_METRICS_DB
os.environ['GLOBAL_BLOCK_METRICS_DB'] = GLOBAL_BLOCK_METRICS_DB
os.environ['PARSER_URL'] = PARSER_URL
os.environ['GLOBAL_CLUSTER_DB'] = GLOBAL_CLUSTER_DB
os.environ['GLOBAL_TASK_DB_SERVICE'] = GLOBAL_TASK_DB_SERVICE
os.environ['COMPONENT_REGISTRY_SERVICE'] = COMPONENT_REGISTRY_SERVICE
os.environ['GLOBAL_BLOCKDB_SERVICE'] = GLOBAL_BLOCKDB_SERVICE
os.environ['CLUSTER_SYNC_URL'] = CLUSTER_SYNC_URL
#os.environ['SERVER_URL'] = SERVER_URL

print("✅ Configuration variables set:")
print(f"   • GATEWAY_URL: {GATEWAY_URL}")
print(f"   • CLUSTER_ID: {CLUSTER_ID}")
print(f"   • GLOBAL_CLUSTER_METRICS_DB: {GLOBAL_CLUSTER_METRICS_DB}")
print("\n📝 These variables are now available in both Python and bash cells!")
print("   - In Python: use GATEWAY_URL, CLUSTER_ID, GLOBAL_CLUSTER_METRICS_DB PARSER_URL GLOBAL_CLUSTER_DB GLOBAL_TASK_DB_SERVICE")
print("   - In bash: use $GATEWAY_URL, $CLUSTER_ID, $GLOBAL_CLUSTER_METRICS_DB $PARSER_URL $GLOBAL_CLUSTER_DB $GLOBAL_TASK_DB_SERVICE")
# os.system('echo $GATEWAY_URL')
# os.system('echo $CLUSTER_ID')
# os.system('echo $GLOBAL_CLUSTER_METRICS_DB')
# os.system('echo $PARSER_URL')
# os.system('echo $GLOBAL_CLUSTER_DB')
# os.system('echo $GLOBAL_TASK_DB_SERVICE')
# os.system('echo $COMPONENT_REGISTRY_SERVICE')
# os.system('echo $GLOBAL_BLOCKDB_SERVICE')
# os.system('echo $GLOBAL_BLOCK_METRICS_DB')

### **GATEWAY APIS for Pre-check Policies**

#### **Add add_node policy to Gateway**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/update" \
    -H "Content-Type: application/json" \
    -d '{
            "add_node": {
                "policyRuleURI": "gateway_add_node_policy:1.0.0-stable",
                "parameters": {
                    "min_node_cpu": 1,
                    "min_node_memory": 2048,
                    "min_node_storage": 10000,
                    "required_gpu_models": ["NVIDIA A100", "NVIDIA A6000", "Tesla T4", "NVIDIA L4", "NVIDIA A100-SXM4-80GB"],
                    "min_gpu_memory": 4096,
                    "allowed_tags": ["demo"],
                    "min_network_interfaces": 1,
                    "min_tx_bandwidth": -1,
                    "min_rx_bandwidth": -1
                }
            }
        }' | json_pp

#### **Get all policies of Gateway**:

In [ ]:
%%bash
curl -X GET "http://$GATEWAY_URL/pre-check-policies/get" | json_pp

#### **kubectl Commands**

In [ ]:
%%bash
#to list down pods in services namespace
kubectl get pods  -n services

# To check add_node policies
kubectl logs -f cluster-controller-gateway-9ccc57c88-h4wsl  -n services

# Or Use Grafana Dashboard to check the logs
# http://MANAGEMENTMASTER:32199/
# Go to Explore --> Choose Loki --> select pod(in Label Filters) and give the cluster-controller-gateway-9ccc57c88-h4wsl as value and run Live (if needed Cancel the Live and and run again)

#### **Dry-run of add_node policy from Gateway**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/add_node/dry_run/$CLUSTER_ID" \
    -H "Content-Type: application/json" \
    -d '{
  "id": "demo-worker-node5",
  "memory": 52205,
  "swap": 0,
  "vcpus": {
    "count": 3
  },
  "storage": {
    "disks": 1,
    "size": 125896
  },
  "network": {
    "interfaces": 7,
    "rxBandwidth": 0,
    "txBandwidth": 10000
  },
  "gpus": {
    "count": 2,
    "memory": 30720,
    "modelNames": [
      "Tesla T4"
    ],
    "features": [
      "fp16",
      "tensorcore"
    ],
    "gpus": [
      {
        "modelName": "Tesla T4",
        "memory": 15360
      },
      {
        "modelName": "Tesla T4",
        "memory": 15360
      }
    ]
  },
  "tags": [
    "demo",
    "production"
  ],
  "nodeMetadata": {
    "location": "Rack 2 - DC1",
    "notes": "Installed 2024-12",
    "vendor": "Supermicro"
  }
}' | json_pp

#### **Remove add_node policy from Gateway**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/update" \
    -H "Content-Type: application/json" \
    -d '{
        "add_node": ""
    }'

### **CLUSTER CONTROLLER API's for Pre-check Policies**

#### **Add add_node policy to Cluster Controller**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/update?cluster_id=$CLUSTER_ID" \
    -H "Content-Type: application/json" \
    -d '{
            "add_node": {
                "policyRuleURI": "demo_cluster_controller_add_node_policy:1.0.0-stable",
                "parameters": {
                    "min_node_cpu": 4,
                    "min_node_memory": 4096,
                    "min_node_storage": 100000,
                    "required_gpu_models": ["NVIDIA A100", "NVIDIA A6000", "Tesla T4", "NVIDIA L4"],
                    "min_gpu_memory": 8000,
                    "allowed_tags": ["demo"],
                    "min_network_interfaces": 1,
                    "min_tx_bandwidth": -1,
                    "min_rx_bandwidth": -1,
                    "max_nodes_per_cluster": 10,
                    "max_gpus_per_cluster": 10,
                    "threshold_cluster_vcpu_utilization_for_add_node": 0.0005
                }
            }
        }' | json_pp

#### **Get all policies of Cluster Controller**:

In [ ]:
%%bash
curl -X GET "http://$GATEWAY_URL/pre-check-policies/get?cluster_id=$CLUSTER_ID" | json_pp

#### **kubectl Commands**

In [ ]:
%%bash
#to list down pods in controllers namespace
kubectl get pods -n controllers

# To check add_node policies
kubectl logs -f membership-server-xxxxxxxxx -n controllers

#### **Dry-run of add_node policy from Cluster Controller**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/add_node/dry_run/$CLUSTER_ID" \
    -H "Content-Type: application/json" \
    -d '{
  "id": "demo-worker-node5",
  "memory": 52205,
  "swap": 0,
  "vcpus": {
    "count": 8
  },
  "storage": {
    "disks": 1,
    "size": 125896
  },
  "network": {
    "interfaces": 7,
    "rxBandwidth": 0,
    "txBandwidth": 10000
  },
  "gpus": {
    "count": 2,
    "memory": 30720,
    "modelNames": [
      "Tesla T4"
    ],
    "features": [
      "fp16",
      "tensorcore"
    ],
    "gpus": [
      {
        "modelName": "Tesla T4",
        "memory": 15360
      },
      {
        "modelName": "Tesla T4",
        "memory": 15360
      }
    ]
  },
  "tags": [
    "demo",
    "production"
  ],
  "nodeMetadata": {
    "location": "Rack 2 - DC1",
    "notes": "Installed 2024-12",
    "vendor": "Supermicro"
  }
}' | json_pp

#### **Remove add_node policy from Cluster Controller**:

In [ ]:
%%bash 
curl -X POST "http://$GATEWAY_URL/pre-check-policies/update?cluster_id=$CLUSTER_ID" \
    -H "Content-Type: application/json" \
    -d '{
        "add_node": ""
    }'

### **Add node to existing cluster using binary**:

- As it will involve manual steps in onboarding a node to cluster, to help with that, we have created a binary `membership-client` which will help you to onboard a node to cluster.
- Steps:
    - Please check Pre-requisites above
    - copy the kubeconfig file from Master node(`$HOME/.kube/config`) to your node at `$HOME/.kube/config`
    - copy the binary to the node you want to onboard
    - give execute permission to the binary: `chmod +x membership-client`
    - run the binary with the command for help: `./membership-client --help`
    - get the `SHA-256 Hash of ca.crt` file from `Master Node` of cluster using:
      ```bash
      openssl x509 -pubkey -in /etc/kubernetes/pki/ca.crt | openssl rsa -pubin -outform der 2>/dev/null | sha256sum | awk '{print $1}'
      ```

#### **Update Cluster CA HASH to Cluster DB(OneTime for the Cluster)**:
- Update the certificate hash to cluster metadata using the command below:

In [ ]:
%%bash
curl -X PUT http://$GLOBAL_CLUSTER_DB/clusters/$CLUSTER_ID -H "Content-Type: application/json" -d '{
   "config.certHash": "yourHASH"
}' | json_pp

#### **Onboard Nodes to Cluster**
- Now you can onboard the node to cluster using the binary: 
    - the command to onboard a node to cluster is:
        ```bash
        sudo ./membership-client --action=join -gpu --api=http://MANAGEMENTMASTER:30600 --cluster-id=gcp-demo-cluster --metadata PathToMetadata.json --tags "demo,production"
        ```
    - Parameters:
        - --api here is Gateway URL
        - --cluster-id is the cluster you want to join
        - -gpu is optional, if you want to onboard a GPU node
        - --action=join, remove, dry-run-add, dry-run-remove
        - --metadata is the path to the metadata file
        - --tags is the tags you want to add to the node

    - **Important**: Once Node joined, you can replace toml file to /etc/containerd/config.toml

#### **Sync the Cluster once**

In [ ]:
%%bash
curl http://$CLUSTER_SYNC_URL/sync-cluster

#### **kubectl Commands**

In [ ]:
%%bash
watch kubectl get nodes
kubectl get nodes

#### **Check Cluster Data and Metrics**:

In [ ]:
%%bash
curl -X GET http://$GATEWAY_URL/clusters/read/$CLUSTER_ID | json_pp

In [ ]:
%%bash
curl -X GET http://$GATEWAY_URL/cluster-metrics/$CLUSTER_ID -H "Content-Type: application/json" | json_pp

### **BLOCK HEALTH CHECK POLICY**([Doc](https://github.com/OpenCyberspace/OpenOS.AI-Documentation/blob/main/block/block.md#block-health-checker))

- Policy monitors the health of all the instances
    - If not healthy, then call the reassign API to reassign the block to another node
    - You can call any other API to notify the block owner or admin etc
- check in **[policies/block_healthcheck_policy](policies/block_healthcheck_policy)**
- **Register health check policy(Stability Check) for Block**:

    - Upload the Zip file to AIOS Policy Storage:
    - Register the policy:

In [ ]:
%%bash
bash policies/block_healthcheck_policy/upload.sh

In [ ]:
%%bash 
bash policies/block_healthcheck_policy/register.bash

### **RESOURCE ALLOCATOR POLICY for Block Allocation([Doc](https://docs.aigr.id/cluster-controller/cluster-controller/#resource-allocator-policy))**

- 4 actions in resource allocator policy:
    - `dry_run`: Simulate the allocation of a block to a node without actually allocating it
    - `allocation`: Allocate instance of a block to a node
    - `scale`: Upscale or Downscale the instance of a block on a node
    - `reassignment`: Reassign the instance of a block to another node
    
- Added more checks in Reassignment action of the policy
- check in **[policies/resource_allocator](policies/resource_allocator)**
- **Register Resource Allocation Policy for Block**:

    - Upload the Zip file to AIOS Policy Storage:
    - Register the policy:

In [ ]:
%%bash
bash policies/resource_allocator/upload.sh

In [ ]:
%%bash 
bash policies/resource_allocator/register.bash

### **CRETE BLOCKS in Cluster(With Advanced Policies)**

#### **Register Components**

In [ ]:
%%bash 
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
  -H "Content-Type: application/json" \
  -d @./Part-2/demo_block/component_demo_magistral.json | json_pp

#### **Unregister Components**

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.demo-magistral-llama_cpp:1.0.0-stable"}' | json_pp

#### **Deploy Block**

In [ ]:
%%bash
curl -X POST http://$PARSER_URL/api/createBlock \
 -H "Content-Type: application/json" \
 -d @./Part-2/demo_block/allocation-demo-mistral.json | json_pp

#### **kubectl Commands (in master node of Cluster or Control Plane)**

In [ ]:
%%bash
#to list down all namespaces
kubectl get namespaces

#to list down pods in blocks namespace
kubectl get pods -n blocks

#to get the log of a specific pod in blocks namespace
# health, proxy, block-executor, redis, mapper, are the containers in block
kubectl logs -f <pod-name> -n blocks health

# instance and redis are the containers in instances
kubectl logs -f <pod-name> -n blocks instance

#to get the log of resource allocator policy pod in controllers namespace
# redis, infra, block-transactions, parameter-updater, monitor, health are the containers in controller
kubectl logs -f <pod-name> -n controllers infra

#### **Get Block Details**

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCKDB_SERVICE/blocks/demo-magistral-block-llama-cpp \
    -H "Content-Type: application/json" | json_pp

#### **Create Inference Server For The Cluster**
- Run this commands in your cluster node(like master node)
- `kubectl create namespace inference-server`
- `kubectl create -f inference_server/inference_server.yaml`
#### **Do the Inference**

In [ ]:
%%bash
curl -X POST  http://DEMOCLUSTERMASTER:31504/v1/infer \
  -H "Content-Type: application/json" \
  -d '{
  "model": "demo-magistral-block-llama-cpp",
  "session_id": "session-2",
  "seq_no": 16,
  "data": {
    "mode": "chat",
    "gen_params": {
      "temperature": 0.1,
      "top_p": 0.95,
      "max_tokens": 4096
    },
    "message": "Give me code for adding two integers list element wise in python",
    "system_message": "You are a helpful assistant that provides code examples."
  },
  "graph": {},
  "selection_query": {
    
  }
}'

#### **Block Metrics**

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/demo-magistral-block-llama-cpp | json_pp

#### **Remove Block**

In [ ]:
%%bash
curl -X POST http://$GATEWAY_URL/controller/removeBlock/gcp-demo-cluster \
    -H "Content-Type: application/json" \
    -d '{"block_id": "demo-magistral-block-llama-cpp"}'